# 3 · One economy, two file formats: Dynare compatibility

**The story.** Economists have written models like our island's in a file format called **Dynare** (`.mod` files) for decades. Dyno reads its native `.dyno` format *and* the Dynare dialect. Here we write the **same island economy** in both formats and check — syntax by syntax, number by number — that the two files agree.

You don't need to know Dynare. Think of it as saving the same spreadsheet as both `.xlsx` and `.ods`, then verifying nothing was lost: if the numbers match after translation, our earlier notebooks' results and the Dynare world's results are the same results.

The supplied [models/rbc.mod](models/rbc.mod) exercises the Dynare syntax that Dyno supports here; it is not a claim that every Dynare feature is supported.

In [1]:
import numpy as np
import pandas as pd
from dyno import DynoModel

pd.set_option("display.precision", 4)

## 1. Load the Dynare version

`DynoModel` detects the format from the file extension (`.mod` here, `.dyno` elsewhere) and produces the same kind of object — same symbols, same equations rendered below, same checks.

In [2]:
mod_model = DynoModel("models/rbc.mod")
mod_model

Component,Count,Symbols
Equations,8,
Endogenous,8,"r, c, n, w, k, i, a, y"
Exogenous,1,epsilon
Parameters,7,"beta, delta, nss, eta, alpha, rho, chi"


## 2. The two dialects, side by side

| Feature | Dynare `.mod` | Native `.dyno` |
| :--- | :--- | :--- |
| Variables | `var c, k, y;` | Inferred from the equations |
| Shocks | `varexo epsilon;` | Inferred from a process assignment |
| Parameter | `beta = 0.985;` | `beta <- 0.985` |
| Timing | `k(-1)`, `c(+1)` | `k[t-1]`, `c[t+1]` |
| Equations | inside `model; ... end;` | a direct list of equations |
| Steady state | `steady_state_model; k=... end;` | `k[~] <- ...` |
| Shock scale | `shocks; var epsilon; stderr 0.009;end;` | `epsilon[t] <- N(0, 0.009)` |

The last row matters most: native `N(mean, standard deviation)` matches Dynare's `stderr` (a standard deviation). The two files use the same value, 0.009 — otherwise the comparison below would fail *because the models genuinely differ*, not because of a bug.

## 3. Check and solve, exactly as with a `.dyno` file

Once imported, the Dynare file rides the same workflow as notebook 1: uniqueness check, first-order solve, same decision-rule table.

In [3]:
mod_model.check()
solution = mod_model.solve()
_, rule = solution.decision_rule.coefficients_as_df()
rule.loc[["a[t]", "y[t]", "c[t]", "i[t]"], ["a[t-1]", "epsilon[t]"]]

,a[t-1],epsilon[t]
a[t],0.9000,1.0000
y[t],1.2188,1.3542
c[t],0.2191,0.2434
i[t],0.9997,1.1107


## 4. The same discovery, told by the Dynare file

Same model, same resting position, same size of discovery — so the impulse response should be the familiar story from notebook 1: technology up about 0.9%, investment springing about eighteen times farther than consumption on impact.

In [4]:
mod_irf = solution.irfs(type="percent", T=40)["epsilon"]
display(mod_irf[["a", "y", "c", "i"]].head())
solution.plot(type="percent", variables=["a", "y", "c", "i"], T=40)

,a,y,c,i
0,0.0000,0.0000,0.0000,0.0000
1,0.9000,1.3918,0.3123,5.7407
2,0.8100,1.2734,0.3571,4.9649
3,0.7290,1.1656,0.3928,4.2787
4,0.6561,1.0673,0.4207,3.6726


alt.Chart(...)

## 5. Does the translation preserve everything?

Now the actual test. Load the native [models/rbc.dyno](models/rbc.dyno), compute both impulse responses, and compare on every variable the two files share (by name — parsers may order lists differently). The assertion fails loudly if the translation drifted anywhere.

In [5]:
native_model = DynoModel("models/rbc.dyno")
native_irf = native_model.solve().irfs(type="percent", T=40)["epsilon"]
variables = sorted(set(mod_irf.columns) & set(native_irf.columns))

steady_state_gap = max(abs(native_model.steady_state[v] - mod_model.steady_state[v]) for v in variables)
irf_gap = np.max(np.abs(native_irf[variables].to_numpy() - mod_irf[variables].to_numpy()))
np.testing.assert_allclose(native_irf[variables], mod_irf[variables], rtol=1e-8, atol=1e-10)
print(f"Maximum steady-state difference: {steady_state_gap:.2e}")
print(f"Maximum IRF difference: {irf_gap:.2e}")

Maximum steady-state difference: 0.00e+00
Maximum IRF difference: 3.39e-15


## 6. Theory versus one drawn history

A solved, verified translation is the perfect base for analysis. One comparison economists reach for first: what the model *promises* about typical fluctuations (its moments, averaged over all dice throws — notebook 1's glossary) versus what *one drawn sample of history* actually shows. The two will not coincide: 200 periods is a small sample of all possible futures.

In [6]:
simulation = solution.simulate(T=200, mode="random", units="level", rng=11)
sample = simulation.to_df()[mod_model.symbols["endogenous"]]

cov_uncond, _ = solution.decision_rule.moments()
model_std = pd.Series(np.sqrt(np.diag(cov_uncond)), index=mod_model.symbols["endogenous"])
model_corr_y = pd.Series(
    cov_uncond[:, mod_model.symbols["endogenous"].index("y")] / model_std.values / model_std.loc["y"],
    index=mod_model.symbols["endogenous"],
)

summary = pd.DataFrame({
    "Std. dev., model": model_std,
    "Std. dev., sample": sample.std(),
    "Corr. with y, model": model_corr_y,
    "Corr. with y, sample": sample.corrwith(sample["y"]),
})
summary.loc[["y", "c", "i", "n"]]

,"Std. dev., model","Std. dev., sample","Corr. with y, model","Corr. with y, sample"
y,0.0122,0.0239,1.0,1.0000
c,0.0022,0.0099,1.0,0.7951
i,0.0100,0.0171,1.0,0.9371
n,0.0024,0.0040,1.0,0.8839


The model promises that consumption, investment and hours all co-move perfectly with output (`Corr. with y, model`), and a 200-period draw confirms the pattern approximately. These are statistics of *this* model and this one sample — not calibrated empirical facts about any real economy.

**Takeaway.** A supported `.mod` file enters the exact same load → check → solve → respond → simulate workflow as a `.dyno` file. The numerical cross-check catches calibration or shock-interpretation differences *before* results get compared; the moments comparison connects the solved model to data-like analysis.